# Fitting the pair source with a normalizing flow

Fits a model of the 3D pair-separation source `S(r)` so that
`C(q) = < K(|q|, |r|, angle(q, r)) >_{r ~ S}` reproduces the tabulated correlation function `65.dat`.
`K` is the kernel `Kl-ppiplus.dat` (a 100x100x100 table in `|q|`, `|r|`, `theta`).

Two source models are trained and compared against the *true* source samples `ppi-0.65.dat`
(500k points `(x, y, z)` in fm):

1. **Gaussian** - `r = L z + mu` (the model of the original notebook).
2. **RealNVP** - affine coupling layers followed by the same affine map, so it contains the Gaussian as a special case.

### Fixes compared with the previous version
* `Cholesky` bijector: the **diagonal of `L` is now forced positive** (softplus). Before, the comment claimed this but the raw
  entries were used, so `log(diag)` could become NaN during training.
* The **shift (mean) was a variable that was never used** (and initialised at `7` while the target mean was `4`). It is now applied in
  `forward`/`inverse`, so the source centre is actually fitted.
* The loss could produce **NaN gradients**: `acos` has an infinite derivative at `+-1` (the clip only protected the value, not the
  gradient), and `q = 0` / `r = 0` gave `0/0`. Now the ratio is clipped to `[-1+1e-6, 1-1e-6]` and magnitudes get a tiny epsilon; `q = 0` is excluded from the grid.
* The unused `lr_decay` is replaced by a cosine learning-rate schedule; the dead `KL_Divergence` duplicate of the loss was removed.

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import time
import numpy as np

# For convenience
tfd = tfp.distributions
tfb = tfp.bijectors
DTYPE=np.float32
import matplotlib.pyplot as plt
plt.rcParams.update({'xtick.top': True, 'ytick.right': True, 'xtick.direction': 'in', 'ytick.direction': 'in',
                     'xtick.minor.visible': True, 'ytick.minor.visible': True})

In [ ]:
class RegularGridInterpolator:
    """
    A reusable nD linear interpolator.
    
    Initializes once with the reference grid and values. The object can then be
    called repeatedly with new query points for fast, graph-based interpolation.
    """
    def __init__(self, x_ref_min, x_ref_max, y_ref):
        # --- ONE-TIME SETUP ---
        # Store grid data as TensorFlow constants to embed them in the graph.
        self.x_ref_min = tf.constant(x_ref_min, dtype=tf.float32)
        self.x_ref_max = tf.constant(x_ref_max, dtype=tf.float32)
        self.y_ref = tf.constant(y_ref, dtype=tf.float32)

    @tf.function
    def __call__(self, x_query):
        """
        Performs interpolation. This method is compiled into a static graph
        by @tf.function for high performance.
        
        Args:
            x_query: A tensor of x-coordinates to interpolate.
        
        Returns:
            A tensor of interpolated y-values.
        """
        # --- REPEATED, FAST EXECUTION ---
        return tfp.math.batch_interp_regular_nd_grid(
            x=x_query, 
            x_ref_min=self.x_ref_min, 
            x_ref_max=self.x_ref_max, 
            y_ref=self.y_ref,
            axis=-3
        )

# --- Usage ---
y_values = np.reshape(np.loadtxt("Kl-ppiplus.dat"), (100, 100, 100))   # axes: |q|, |r|, theta
x_min = [0, 0, 0]
x_max = [100, 40, np.pi]                                                # |q| [MeV/c], |r| [fm], theta [rad]
interpolator = RegularGridInterpolator(x_ref_min=x_min, x_ref_max=x_max, y_ref=y_values)

## Initial guess for the source (same as before)

In [ ]:
init_mean = np.array([4.0, 0.0, -0.5], dtype=np.float32)
init_cov = np.diag([80.0, 30.0, 30.0]).astype(np.float32)
base_distribution = tfd.MultivariateNormalDiag(loc=tf.zeros(3), scale_diag=tf.ones(3))

## Gaussian source: `r = L z + mu` with `L` lower-triangular and positive diagonal

In [ ]:
class Cholesky(tfb.Bijector):
    """y = L x + shift;  L lower triangular with strictly positive diagonal (softplus + 1e-3)."""
    def __init__(self, mean, covariance, name='TrainableCholesky'):
        super().__init__(forward_min_event_ndims=1, validate_args=False, name=name)
        L0 = np.linalg.cholesky(np.asarray(covariance, dtype=np.float64))
        self._tril = np.stack(np.tril_indices(3, -1), axis=1).astype(np.int32)       # (1,0), (2,0), (2,1)
        raw_diag = np.log(np.expm1(np.diag(L0) - 1e-3))                               # inverse softplus
        self._raw_diag = tf.Variable(raw_diag, dtype=DTYPE, name='raw_diag')
        self._off_diag = tf.Variable(L0[np.tril_indices(3, -1)], dtype=DTYPE, name='off_diag')
        self._shift = tf.Variable(np.asarray(mean), dtype=DTYPE, name='shift')

    def _diag(self):
        return tf.math.softplus(self._raw_diag) + 1e-3

    def _L(self):
        return tf.scatter_nd(self._tril, self._off_diag, [3, 3]) + tf.linalg.diag(self._diag())

    def _forward(self, x):
        return tf.linalg.matvec(self._L(), x) + self._shift

    def _inverse(self, y):
        return tf.linalg.triangular_solve(self._L(), (y - self._shift)[..., tf.newaxis], lower=True)[..., 0]

    def _forward_log_det_jacobian(self, x):
        return tf.reduce_sum(tf.math.log(self._diag()))

    def _inverse_log_det_jacobian(self, y):
        return -tf.reduce_sum(tf.math.log(self._diag()))

gauss_bijector = Cholesky(init_mean, init_cov)
gauss_flow = tfd.TransformedDistribution(distribution=base_distribution, bijector=gauss_bijector)
gauss_vars = list(gauss_bijector.trainable_variables)

# sanity check: the untrained model must reproduce the initial guess
s = gauss_flow.sample(200000).numpy()
print('mean', s.mean(0), '\ncov\n', np.cov(s.T))

## Loss: `|| <K>_r - C ||`  (numerically safe version)

In [ ]:
@tf.function
def compute_loss(xs, true_ys, qs, interpolator):
    eps = 1e-6
    r = tf.norm(xs, axis=1) + eps                            # [N]
    q = tf.norm(qs, axis=1) + eps                            # [M]
    dot_products = tf.tensordot(qs, xs, axes=[[1], [1]])     # [M, N]
    ratio = dot_products / tf.tensordot(q, r, axes=0)
    # acos has an infinite slope at +-1: clip *inside* the interval so the gradient stays finite
    angle = tf.math.acos(tf.clip_by_value(ratio, -1.0 + 1e-6, 1.0 - 1e-6))
    q_grid, r_grid = tf.meshgrid(q, r, indexing="ij")
    coords = tf.stack([q_grid, r_grid, angle], axis=-1)      # [M, N, 3]
    mean_ys = tf.math.reduce_mean(interpolator(coords), axis=1)
    return tf.norm(mean_ys - true_ys)

## Data: q grid, target correlation function, and the true source samples

In [ ]:
q_max, Nqs = 50, 20
g = (-q_max + 2 * q_max / Nqs * np.arange(Nqs)).astype(np.float32)
qs = np.stack(np.meshgrid(g, g, g, indexing='ij'), axis=-1).reshape(-1, 3)   # same order as the old triple loop
true_Cq = np.loadtxt("65.dat", dtype=np.float32)
nonzero = np.where(np.linalg.norm(qs, axis=1) > 0)[0]                         # q = 0 has no direction
train_idx = nonzero[::8]                                                       # 1000 q points, as before

true_source = np.loadtxt("ppi-0.65.dat")                                       # 500k samples of the real source [fm]
print('true source: mean', true_source.mean(0).round(2), ' std', true_source.std(0).round(2))

## Training helper

In [ ]:
def train(flow, variables, n_steps=3000, n_samples=1000, lr=2e-3, log_every=250):
    schedule = tf.keras.optimizers.schedules.CosineDecay(lr, n_steps, alpha=0.03)
    optimizer = tf.keras.optimizers.Adam(schedule)
    q_tr, c_tr = tf.constant(qs[train_idx]), tf.constant(true_Cq[train_idx])

    @tf.function
    def step():
        with tf.GradientTape() as tape:
            xs = flow.sample(n_samples)
            loss = compute_loss(xs, c_tr, q_tr, interpolator)
        grads = tape.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))
        return loss

    history = []
    for i in range(n_steps):
        history.append(float(step()))
        if i % log_every == 0 or i == n_steps - 1:
            print(f'step {i:5d}  loss {np.mean(history[-log_every:]):.5f}')
    return np.array(history)

hist_gauss = train(gauss_flow, gauss_vars, n_steps=1500, lr=5e-3)

## RealNVP source

Six affine coupling layers (`y_b = x_b * exp(s(x_a)) + t(x_a)`, `s, t` from a small tanh MLP), with a rotation of the coordinates between
layers, followed by the same trainable `Cholesky` affine map as above. The last layer of every MLP is initialised to zero, so the untrained
model is *exactly* the Gaussian initial guess and training starts from the same loss.

In [ ]:
def build_realnvp(n_layers=6, hidden=32, log_scale_max=1.5):
    layers, variables = [], []
    for k in range(n_layers):
        n_masked = 1 if k % 2 == 0 else 2
        net = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden, activation='tanh'),
            tf.keras.layers.Dense(2 * (3 - n_masked), kernel_initializer='zeros', bias_initializer='zeros')])
        net.build((None, n_masked))
        def shift_and_log_scale_fn(x0, output_units, _net=net, **kwargs):
            shift, raw = tf.split(_net(x0), 2, axis=-1)
            return shift, log_scale_max * tf.math.tanh(raw)            # bounded log-scale keeps training stable
        layers.append(tfb.RealNVP(num_masked=n_masked, shift_and_log_scale_fn=shift_and_log_scale_fn))
        layers.append(tfb.Permute([1, 2, 0]))
        variables += net.trainable_variables
    affine = Cholesky(init_mean, init_cov)
    variables += affine.trainable_variables
    # tfb.Chain applies its list right-to-left, so reverse: first coupling layer acts first, affine map last
    bijector = tfb.Chain([affine] + layers[::-1])
    return tfd.TransformedDistribution(distribution=base_distribution, bijector=bijector), variables

nvp_flow, nvp_vars = build_realnvp()
hist_nvp = train(nvp_flow, nvp_vars, n_steps=3000, lr=2e-3)

## Compare the fitted sources with the real one

In [ ]:
n_eval = 50000
models = {'Gaussian fit': gauss_flow.sample(n_eval).numpy(), 'RealNVP fit': nvp_flow.sample(n_eval).numpy()}
colors = {'true source (ppi-0.65)': 'k', 'Gaussian fit': 'tab:blue', 'RealNVP fit': 'tab:red'}
samples = {'true source (ppi-0.65)': true_source[:n_eval], **models}

def correlation_np(src, q_points, chunk=250):
    out = []
    for i in range(0, len(q_points), chunk):
        xs, qc = tf.constant(src, tf.float32), tf.constant(q_points[i:i + chunk])
        r = tf.norm(xs, axis=1) + 1e-6; q = tf.norm(qc, axis=1) + 1e-6
        ratio = tf.tensordot(qc, xs, axes=[[1], [1]]) / tf.tensordot(q, r, axes=0)
        ang = tf.math.acos(tf.clip_by_value(ratio, -1 + 1e-6, 1 - 1e-6))
        qg, rg = tf.meshgrid(q, r, indexing='ij')
        out.append(tf.reduce_mean(interpolator(tf.stack([qg, rg, ang], -1)), axis=1).numpy())
    return np.concatenate(out)

print(f"{'source':26s} {'rms(C - C_65)':>14s}   mean                    std")
for name, s in samples.items():
    c = correlation_np(s, qs[nonzero])
    print(f"{name:26s} {np.sqrt(np.mean((c - true_Cq[nonzero])**2)):14.5f}   {s.mean(0).round(2)}   {s.std(0).round(2)}")

fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
for i, label in enumerate(['x (out) [fm]', 'y (side) [fm]', 'z (long) [fm]']):
    bins = np.linspace(-45, 60, 100)
    for name, s in samples.items():
        ax[i].hist(s[:, i], bins=bins, density=True, histtype='step', color=colors[name], lw=1.8, label=name)
    ax[i].set_yscale('log'); ax[i].set_ylim(1e-5, None); ax[i].set_xlabel(label)
for name, s in samples.items():
    ax[3].hist(np.linalg.norm(s, axis=1), bins=np.linspace(0, 60, 100), density=True, histtype='step', color=colors[name], lw=1.8)
ax[3].set_xlabel('|r| [fm]'); ax[0].set_ylabel('density'); ax[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()